In [331]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing   import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_auc_score  
from statsmodels import api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

dane = pd.read_csv('C:/Users/kubap/Desktop/III rok/semestr VI/python/projekt/alzheimers_disease_data.csv')
dane = dane.iloc[:, 1:-1]
mapowanie_pochodzenia = {
    0: 'Caucasian',
    1: 'African American',
    2: 'Asian',
    3: 'Other'
}
dane['Ethnicity'] = dane['Ethnicity'].map(mapowanie_pochodzenia)

mapowanie_edukacji = {
    0: 'None',
    1: 'High School',
    2: "Bachelor's",
    3: 'Higher'
}
dane['EducationLevel'] = dane['EducationLevel'].map(mapowanie_edukacji)

kopia_danych = dane.copy()
    

In [332]:
for c in dane.select_dtypes("object").columns:
    print(dane[c].value_counts(True))

Ethnicity
Caucasian           0.594695
African American    0.211261
Other               0.098185
Asian               0.095859
Name: proportion, dtype: float64
EducationLevel
High School    0.397394
Bachelor's     0.295952
None           0.207538
Higher         0.099116
Name: proportion, dtype: float64


C:\Users\kubap\AppData\Local\Temp\ipykernel_20780\584027146.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for c in dane.select_dtypes("object").columns:


In [333]:
print(dane.isnull().any())
print(dane.isnull().sum().sum())

Age                          False
Gender                       False
Ethnicity                    False
EducationLevel               False
BMI                          False
Smoking                      False
AlcoholConsumption           False
PhysicalActivity             False
DietQuality                  False
SleepQuality                 False
FamilyHistoryAlzheimers      False
CardiovascularDisease        False
Diabetes                     False
Depression                   False
HeadInjury                   False
Hypertension                 False
SystolicBP                   False
DiastolicBP                  False
CholesterolTotal             False
CholesterolLDL               False
CholesterolHDL               False
CholesterolTriglycerides     False
MMSE                         False
FunctionalAssessment         False
MemoryComplaints             False
BehavioralProblems           False
ADL                          False
Confusion                    False
Disorientation      

## Wykrywanie outlierów

In [334]:
ilosciowe = ['Age', 'BMI', 'AlcoholConsumption','PhysicalActivity', 'DietQuality', 'SleepQuality', 'SystolicBP', 'DiastolicBP', 'CholesterolTotal', 
           'CholesterolLDL', 'CholesterolHDL', 'CholesterolTriglycerides', 'MMSE', 'FunctionalAssessment', 'ADL']
Q1 = dane[ilosciowe].quantile(0.25)
Q3 = dane[ilosciowe].quantile(0.75)
laczone = pd.concat([Q1, Q3], axis=1)
# print(laczone)
IQR = Q3 - Q1
IQR
dolna_granica = (Q1 - 1.5 * IQR).clip(lower=0) #poniewaz zadna z tych cech nie moze byc ujemna, to dolna granica rowniez nie moze byc ujemna
gorna_granica = Q3 + 1.5 * IQR
granice = pd.concat([dolna_granica, gorna_granica], axis=1, keys=['Dolna Granica', 'Gorna Granica'])
granice

,Dolna Granica,Gorna Granica
Age,43.000000,107.000000
BMI,3.223853,52.257332
AlcoholConsumption,0.000000,30.185112
PhysicalActivity,0.000000,14.713807
DietQuality,0.000000,15.208879
SleepQuality,0.863712,13.181807
SystolicBP,44.500000,224.500000
DiastolicBP,27.500000,151.500000
CholesterolTotal,82.584923,369.699697
CholesterolLDL,0.000000,273.540635


In [335]:
maska_outliers = (dane[ilosciowe] < dolna_granica) | (dane[ilosciowe] > gorna_granica)
outliers = dane[maska_outliers.any(axis=1)]
print(maska_outliers.sum()) #liczba outlierow w kazdej z cech
#Jako, ze liczba outlierow dla kazdej z cech byla rowna 0, sprawdzamy czy na pewno wszystko dobrze dziala poprzez wprowadzenie sztucznego outliera
kopia_danych.loc[20, 'Age'] = 150
maska_test = (kopia_danych[ilosciowe] < dolna_granica) | (kopia_danych[ilosciowe] > gorna_granica)
print(maska_test.sum())

Age                         0
BMI                         0
AlcoholConsumption          0
PhysicalActivity            0
DietQuality                 0
SleepQuality                0
SystolicBP                  0
DiastolicBP                 0
CholesterolTotal            0
CholesterolLDL              0
CholesterolHDL              0
CholesterolTriglycerides    0
MMSE                        0
FunctionalAssessment        0
ADL                         0
dtype: int64
Age                         1
BMI                         0
AlcoholConsumption          0
PhysicalActivity            0
DietQuality                 0
SleepQuality                0
SystolicBP                  0
DiastolicBP                 0
CholesterolTotal            0
CholesterolLDL              0
CholesterolHDL              0
CholesterolTriglycerides    0
MMSE                        0
FunctionalAssessment        0
ADL                         0
dtype: int64


## Kodowanie i skalowanie zmiennych

In [336]:
nominalne = ['Ethnicity', 'EducationLevel']
dane = pd.get_dummies(dane, columns=nominalne, drop_first=True)

In [337]:
scaler = StandardScaler()
dane[ilosciowe] = scaler.fit_transform(dane[ilosciowe])

## Dobór zmiennych objaśniających

In [ ]:
y = dane['Diagnosis']
x = dane.drop('Diagnosis', axis=1)

In [ ]:
#dodajemy stala do zmiennych objasniajacych, aby model mial punkt odniesienia
x = sm.add_constant(x)

In [ ]:
print("Typy danych w x:")
print(x.dtypes)
print("\nTyp danych w y:")
print(y.dtype)

Typy danych w x:
const                         float64
Age                           float64
Gender                          int64
BMI                           float64
Smoking                         int64
AlcoholConsumption            float64
PhysicalActivity              float64
DietQuality                   float64
SleepQuality                  float64
FamilyHistoryAlzheimers         int64
CardiovascularDisease           int64
Diabetes                        int64
Depression                      int64
HeadInjury                      int64
Hypertension                    int64
SystolicBP                    float64
DiastolicBP                   float64
CholesterolTotal              float64
CholesterolLDL                float64
CholesterolHDL                float64
CholesterolTriglycerides      float64
MMSE                          float64
FunctionalAssessment          float64
MemoryComplaints                int64
BehavioralProblems              int64
ADL                           flo

In [ ]:
#konwertujemy dane na float, zeby statsmodels mogl dzialac
x = x.astype(float)

In [ ]:
vif_tabela = pd.DataFrame(index = x.columns, columns=['VIF'])
vif_tabela['VIF'] = [variance_inflation_factor(x.values, i) for i in range(x.shape[1])]
vif_tabela
max(vif_tabela['VIF'][1:]) #pomijamy VIF dla constans, bo on zawsze bedzie bardzo duzy. Nie ma zadnej cechy, ktora ma VIF wiekszy niz 10, wiec nie musimy sie martwic o wspoliniowosc. 

1.562549538128679

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.3, random_state=42) 

In [ ]:
C_values = [0.05, 0.1, 1, 10, 100]

for C in C_values:
    model = LogisticRegression(
        l1_ratio = 1 ,
        solver = 'liblinear',
        C = C,
        max_iter = 1000
    )
    
    model.fit(X_train, y_train)
    
    wspolczynniki = model.coef_[0]
    num_zero = np.sum(wspolczynniki == 0)
    
    y_prob = model.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, y_prob)

    print(f"C={C} → liczba wyzerowanych zmiennych: {num_zero}, AUC: {auc}")

C=0.05 → liczba wyzerowanych zmiennych: 28, AUC: 0.8748518049139447
C=0.1 → liczba wyzerowanych zmiennych: 23, AUC: 0.8802481501165121
C=1 → liczba wyzerowanych zmiennych: 2, AUC: 0.8791341318834063
C=10 → liczba wyzerowanych zmiennych: 0, AUC: 0.8780201136503004
C=100 → liczba wyzerowanych zmiennych: 0, AUC: 0.8779179101426761


In [ ]:
model = LogisticRegression(
    l1_ratio = 1,
    solver = 'liblinear',
    C = 0.1,
    max_iter = 1000
)
model.fit(X_train, y_train)
y_prob = model.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, y_prob)

wspolczynniki = model.coef_[0]
wybrane_zmienne = x.columns[wspolczynniki != 0]
odrzucone_zmienne = x.columns[wspolczynniki == 0]



In [ ]:
selekcja = pd.DataFrame(index = x.columns, columns = ['Czy wybrana?'])
maska = selekcja.index.isin(wybrane_zmienne)
selekcja['Czy wybrana?'] = np.where(maska, 'Tak', 'Nie')
selekcja

,Czy wybrana?
const,Tak
Age,Tak
Gender,Nie
BMI,Nie
Smoking,Nie
AlcoholConsumption,Tak
PhysicalActivity,Tak
DietQuality,Tak
SleepQuality,Tak
FamilyHistoryAlzheimers,Nie


In [ ]:
wynik = pd.DataFrame(index=wybrane_zmienne)
wynik['wspolczynniki'] = wspolczynniki[wspolczynniki != 0]
wynik


,wspolczynniki
const,-0.748284
Age,-0.026089
AlcoholConsumption,-0.010219
PhysicalActivity,-0.022138
DietQuality,0.024824
SleepQuality,-0.044510
DiastolicBP,0.089628
CholesterolHDL,0.031221
CholesterolTriglycerides,0.049396
MMSE,-0.764107


In [340]:

sm_model = sm.Logit(y_train, X_train[wybrane_zmienne]).fit()
print(sm_model.summary())


Optimization terminated successfully.
         Current function value: 0.349210
         Iterations 7
                            Logit Regression Results                           
Dep. Variable:               Diagnosis   No. Observations:                 1504
Model:                           Logit   Df Residuals:                     1490
Method:                            MLE   Df Model:                           13
Date:              niedz., 26 kwi 2026   Pseudo R-squ.:                  0.4570
Time:                         17:02:57   Log-Likelihood:                -525.21
converged:                        True   LL-Null:                       -967.16
Covariance Type:             nonrobust   LLR p-value:                1.441e-180
                               coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------
const                       -2.2117      0.129    -17.200      0.000    

In [ ]:
p_values = sm_model.pvalues
p_values.sort_values(ascending=False) #Jak widac, czesc zmiennych nie jest istotna statystycznie. Sprawdzamy wiec czy po ich usunieciu model bedzie mial lepsze parametry.

AlcoholConsumption          4.411726e-01
DietQuality                 3.327686e-01
PhysicalActivity            2.725415e-01
CholesterolHDL              2.598216e-01
Age                         2.171271e-01
SleepQuality                1.964707e-01
CholesterolTriglycerides    1.690826e-01
DiastolicBP                 4.001488e-02
BehavioralProblems          3.046836e-28
MMSE                        1.355714e-29
MemoryComplaints            1.053648e-40
ADL                         6.101438e-43
FunctionalAssessment        2.801862e-49
const                       2.637501e-66
dtype: float64

In [ ]:
y_pred = model.predict(X_test)
y_pred
confusion_matrix(y_test, y_pred)
accuracy_score(y_test, y_pred)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.83      0.89      0.86       401
           1       0.80      0.69      0.74       244

    accuracy                           0.82       645
   macro avg       0.81      0.79      0.80       645
weighted avg       0.82      0.82      0.81       645



In [ ]:
testowe = p_values[p_values <= 0.05].index
model.fit(X_train[testowe], y_train)
y_pred = model.predict(X_test[testowe])
confusion_matrix(y_test, y_pred)
accuracy_score(y_test, y_pred)
print(classification_report(y_test, y_pred)) 

              precision    recall  f1-score   support

           0       0.83      0.90      0.86       401
           1       0.80      0.70      0.75       244

    accuracy                           0.82       645
   macro avg       0.82      0.80      0.81       645
weighted avg       0.82      0.82      0.82       645



In [ ]:
y_prob = model.predict_proba(X_test[testowe])[:, 1]
auc_po_usunieciu = roc_auc_score(y_test, y_prob)
print(f"AUC po usunieciu zmiennych: {auc_po_usunieciu}, AUC przed usunieciem zmiennych: {auc})")


AUC po usunieciu zmiennych: 0.8802277094149872, AUC przed usunieciem zmiennych: 0.8802583704672745)


## Jako ze po usunieciu zmiennych, ktore sa nieistotne statystycznie, AUC zmniejszyl sie minimalnie, a precyzja odrobine sie poprawila, wybieramy wlasnie ten model (bo ma mniej zmiennych)


In [ ]:
X_test = X_test[testowe]


In [ ]:
oszacowanie = pd.DataFrame(index=testowe)
oszacowanie['Wspolczynniki'] = model.coef_[0]   
oszacowanie

,Wspolczynniki
const,-0.645724
DiastolicBP,0.092549
MMSE,-0.766382
FunctionalAssessment,-1.138568
MemoryComplaints,2.049659
BehavioralProblems,1.683891
ADL,-1.036497
